<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# Uber Pickups 🚕

## Company's Description 📇

<a href="http://uber.com/" target="_blank">Uber</a> is one of the most famous startup in the world. It started as a ride-sharing application for people who couldn't afford a taxi. Now, Uber expanded its activities to Food Delivery with <a href="https://www.ubereats.com/fr-en" target="_blank">Uber Eats</a>, package delivery, freight transportation and even urban transportation with <a href="https://www.uber.com/fr/en/ride/uber-bike/" target="_blank"> Jump Bike</a> and <a href="https://www.li.me/" target="_blank"> Lime </a> that the company funded.


The company's goal is to revolutionize transportation accross the globe. It operates now on about 70 countries and 900 cities and generates over $14 billion revenue! 😮


## Project 🚧

One of the main pain point that Uber's team found is that sometimes drivers are not around when users need them. For example, a user might be in San Francisco's Financial District whereas Uber drivers are looking for customers in Castro.  

(If you are not familiar with the bay area, check out <a href="https://www.google.com/maps/place/San+Francisco,+CA,+USA/@37.7515389,-122.4567213,13.43z/data=!4m5!3m4!1s0x80859a6d00690021:0x4a501367f076adff!8m2!3d37.7749295!4d-122.4194155" target="_blank">Google Maps</a>)

Eventhough both neighborhood are not that far away, users would still have to wait 10 to 15 minutes before being picked-up, which is too long. Uber's research shows that users accept to wait 5-7 minutes, otherwise they would cancel their ride.

Therefore, Uber's data team would like to work on a project where **their app would recommend hot-zones in major cities to be in at any given time of day.**  

## Goals 🎯

Uber already has data about pickups in major cities. Your objective is to create algorithms that will determine where are the hot-zones that drivers should be in. Therefore you will:

* Create an algorithm to find hot zones
* Visualize results on a nice dashboard

## Scope of this project 🖼️

To start off, Uber wants to try this feature in New York city. Therefore you will only focus on this city. Data can be found here:

👉👉<a href="https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+non+Supervis%C3%A9/Projects/uber-trip-data.zip" target="_blank"> Uber Trip Data</a> 👈👈

**You only need to focus on New York City for this project**

## Helpers 🦮

To help you achieve this project, here are a few tips that should help you:

### Clustering is your friend

Clustering technics are a perfect fit for the job. Think about it, all the pickup locations can be gathered into different clusters. You can then use **cluster coordinates to pin hot zones** 😉
    

### Create maps with `plotly`

Check out <a href="https://plotly.com/" target="_blank">Plotly</a> documentation, you can create maps and populate them easily. Obviously, there are other libraries but this one should do the job pretty well.


### Start small grow big

Eventhough Uber wants to have hot-zones per hour and per day of week, you should first **start small**. Pick one day at a given hour and **then start to generalize** your approach.

## Deliverable 📬

To complete this project, your team should:

* Have a map with hot-zones using any python library (`plotly` or anything else).
* You should **at least** describe hot-zones per day of week.
* Compare results with **at least** two unsupervised algorithms like KMeans and DBScan.

Your maps should look something like this:

<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/images/Clusters_uber_pickups.png" alt="Uber Cluster Map" />

In [144]:
#!pip install -q xgboost
#!pip install -q s3fs
 #!pip install -U kaleido

# Load in our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff
import plotly.colors as pc

# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')

# datasets
from sklearn.datasets import load_sample_image, load_digits

# pipeline transfomers
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# split
from sklearn.model_selection import train_test_split, GridSearchCV

# regression
from sklearn.linear_model import LogisticRegression, Ridge

# arbres de décision
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, VotingRegressor, StackingClassifier,  StackingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor, XGBClassifier

# metrics
from sklearn.metrics import r2_score, accuracy_score, silhouette_score, confusion_matrix

### ML non supervisé
from sklearn.cluster import KMeans, MiniBatchKMeans

# montage drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [145]:
print("Loading dataset...")
# url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+non+Supervis%C3%A9/Projects/uber-trip-data.zip"
df = pd.read_csv("/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-apr14.csv")
print("...Done.\n")

print("Number of rows:", len(df))
df.head()

Loading dataset...
...Done.

Number of rows: 564516


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


## Basic exploring and cleaning
1. Display basic statistics about the dataset.

In [146]:
# Step — Display basic statistics

print("\nBasic statistics:\n")
display(df.describe(include="all"))

print("\nPercentage of missing values:\n")
display(df.isna().mean() * 100)


Basic statistics:



,Date/Time,Lat,Lon,Base
count,564516,564516.000000,564516.000000,564516
unique,41999,NaN,NaN,5
top,4/7/2014 20:21:00,NaN,NaN,B02682
freq,97,NaN,NaN,227808
mean,NaN,40.740005,-73.976817,NaN
std,NaN,0.036083,0.050426,NaN
min,NaN,40.072900,-74.773300,NaN
25%,NaN,40.722500,-73.997700,NaN
50%,NaN,40.742500,-73.984800,NaN
75%,NaN,40.760700,-73.970000,NaN



Percentage of missing values:



,0
Date/Time,0.0
Lat,0.0
Lon,0.0
Base,0.0


2. Prétraitement temporel (indispensable)

In [147]:
df["Date/Time"] = pd.to_datetime(df["Date/Time"])
df["weekday"] = df["Date/Time"].dt.day_name()
df["hour"] = df["Date/Time"].dt.hour

df[["Date/Time", "weekday", "hour"]].head()


,Date/Time,weekday,hour
0,2014-04-01 00:11:00,Tuesday,0
1,2014-04-01 00:17:00,Tuesday,0
2,2014-04-01 00:21:00,Tuesday,0
3,2014-04-01 00:28:00,Tuesday,0
4,2014-04-01 00:33:00,Tuesday,0


Commençons petit : un jour + une heure

Exemple conformement aux consignes : lundi 18h.

In [148]:
df_focus = df[(df["weekday"] == "Monday") & (df["hour"] == 18)].copy()
print(df_focus.shape)


(4725, 6)


4️. Préparation des features spatiales

In [149]:


X = df_focus[features]


Elbow method

In [150]:
inertia_values =[]
K_range = range(1, 11)

for k in K_range:
  kmeans = KMeans(n_clusters=k, random_state=42)
  kmeans.fit(X)
  inertia_values.append(kmeans.inertia_)
  print(inertia_values)

[16.75497508541803]
[16.75497508541803, 9.998986533300648]
[16.75497508541803, 9.998986533300648, 7.567073161124358]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617, 3.482699603155296]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617, 3.482699603155296, 3.151287578117601]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617, 3.482699603155296, 3.151287578117601, 2.8776845976468475]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617, 3.482699603155296, 3.151287578117601, 2.8776845976468475, 2.536117426447456]
[16.75497508541803, 9.998986533300648, 7.567073161124358, 6.903748215984715, 4.863099233674617, 3.482699603155296, 3.151287578117601, 

In [151]:
df_elbow= pd.DataFrame({"K": list(K_range), "inertia": inertia_values})
df_elbow.head()

,K,inertia
0,1,16.754975
1,2,9.998987
2,3,7.567073
3,4,6.903748
4,5,4.863099


In [152]:
fig=px.line(
    df_elbow,
    x="K",
    y="inertia",
    markers=True,
    title="Elbow method"
)
fig.update_layout(
    xaxis_title="Number of clusters",
    yaxis_title="inertia (WCSS)"
)
fig.show()


Silhouette method

In [153]:
silhouette_values=[]
K_range = range(2, 11)

for k in K_range:
  kmeans = KMeans(n_clusters=k, random_state=42)
  labels = kmeans.fit_predict(X)
  score = silhouette_score(X, labels)
  silhouette_values.append(score)
  print(silhouette_values)

[np.float64(0.7698882368531464)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042121066)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042121066), np.float64(0.459963219279585)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042121066), np.float64(0.459963219279585), np.float64(0.5122820549307369)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042121066), np.float64(0.459963219279585), np.float64(0.5122820549307369), np.float64(0.4061127461368753)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042121066), np.float64(0.459963219279585), np.float64(0.5122820549307369), np.float64(0.4061127461368753), np.float64(0.5226417880118104)]
[np.float64(0.7698882368531464), np.float64(0.44433498862469295), np.float64(0.44928354042

In [154]:
df_silhouette= pd.DataFrame({"K": list(K_range), "silhouette": silhouette_values})
df_silhouette.head()

fig= px.line(
    df_silhouette,
    x="K",
    y="silhouette",
    markers=True,
    title="Silhouette method"
)
fig.update_layout(
    xaxis_title="Number of clusters",
    yaxis_title="silhouette"
)
fig.show()

En clustering, le meilleur K n’est jamais purement mathématique.

Le choix doit toujours être guidé par l’usage final.
elbow dit : Au-delà de 4–6 clusters => ajout de complexité inutile
silhouette dit K=2 pour des clusters très propres mathématiquement
=> (ici on serait tenté de prendre K=2 mais => Manhattan vs NYC : aucun intérêt d'un point de vue métier choisir k=2 serait totalement inutile pour guider les chauffeurs)

Bonne décision :
-  ni trop peu
- ni trop beaucoup
=> K=6 ou K=7 est un excellent compromis

7. Next, we will take $K=7$ clusters. Apply the KMeans to your dataset.

In [155]:
kmeans = KMeans(n_clusters=7, random_state=42)
labels = kmeans.fit_predict(X)
labels[:10]

array([6, 6, 2, 2, 6, 6, 2, 2, 1, 6], dtype=int32)

# MVP

In [156]:
df_focus['Cluster_KMeans'] = labels + 1  # +1 pour éviter que kmeans assigne par défaut des labels 0 aux clusters
df_focus.head()

,Date/Time,Lat,Lon,Base,weekday,hour,Cluster_KMeans
8638,2014-04-07 18:00:00,40.7407,-73.9915,B02512,Monday,18,7
8639,2014-04-07 18:00:00,40.7514,-73.9940,B02512,Monday,18,7
8640,2014-04-07 18:00:00,40.7632,-73.9758,B02512,Monday,18,3
8641,2014-04-07 18:01:00,40.7623,-73.9754,B02512,Monday,18,3
8642,2014-04-07 18:01:00,40.7553,-73.9868,B02512,Monday,18,7


6️. Carte Plotly des hot-zones

In [157]:
fig = px.scatter_mapbox(
    df_focus,
    lat="Lat",
    lon="Lon",
    color="Cluster_KMeans",
    zoom=10,
    height=600,
    title="Uber Pickups – Hot Zones (April 2014, Monday 18h, KMeans K=7)",
    mapbox_style="carto-positron",
)

fig.show()


In [158]:
centroids = (
    df_focus
    .groupby("Cluster_KMeans")[["Lat", "Lon"]]
    .mean()
    .reset_index()
)

fig.add_scattermapbox(
    lat=centroids["Lat"],
    lon=centroids["Lon"],
    mode="markers",
    marker=dict(size=14, color="black"),
    name="Hot zones (centroids)"
)

fig.show()


In [159]:
fig = px.scatter_3d(
    df_focus,
    x="Lon",
    y="Lat",
    z="hour",
    color="Cluster_KMeans",
    title="Uber Pickups – Spatial + Temporal View",
)
fig.show()


# Pipeline pour 4 créneaux d'affluence :
- jeudi 22h
- vendredi 22h
- samedi 22h
- dimanche 20h

In [160]:
# weekdays =[]
# hours = [ ]
time_slots = [
    {"label": "Thursday_22h", "weekday": "Thursday", "hour": 22}, # approche dynamique avec hour for i in range(0, )
    {"label": "Friday_22h",   "weekday": "Friday",   "hour": 22},
    {"label": "Saturday_22h", "weekday": "Saturday", "hour": 22},
    {"label": "Sunday_20h",   "weekday": "Sunday",   "hour": 20},
]


In [161]:
def process_time_slot(df, weekday, hour, label, k=7):
    # 1. Filtrage temporel
    df_focus = df[(df["weekday"] == weekday) & (df["hour"] == hour)].copy()

    if df_focus.empty:
        print(f"{label} → aucun pickup")
        return None

    # 2. Clustering
    kmeans = KMeans(n_clusters=k, random_state=42)
    df_focus["Cluster_KMeans"] = kmeans.fit_predict(df_focus[features]) + 1

    # 3. Centroids
    centroids = (
        df_focus
        .groupby("Cluster_KMeans")[features] #["lat"] ["lon"]
        .mean()
        .reset_index()
    )

    # 4. Carte des clusters
    fig_map = px.scatter_mapbox(
        df_focus,
        lat="Lat",
        lon="Lon",
        color="Cluster_KMeans",
        zoom=10,
        height=600,
        title=f"Uber Hot-Zones – {label}",
        mapbox_style="carto-positron",
    )

    fig_map.add_scattermapbox(
        lat=centroids["Lat"],
        lon=centroids["Lon"],
        mode="markers",
        marker=dict(size=14, color="black"),
        name="Hot zones (centroids)"
    )

    # 5. Scatter 3D
    fig_3d = px.scatter_3d(
        df_focus,
        x="Lon",
        y="Lat",
        z="hour",
        color="Cluster_KMeans",
        title=f"Spatial + Temporal View – {label}",
    )

    return {
        "label": label,
        "df": df_focus,
        "centroids": centroids,
        "map": fig_map,
        "scatter_3d": fig_3d,
    }


lancement du pipeline

In [162]:
results = []

for slot in time_slots:
    output = process_time_slot(
        df=df,
        weekday=slot["weekday"],
        hour=slot["hour"],
        label=slot["label"],
        k=7
    )
    if output:
        results.append(output)


affichage

In [163]:
color_scale = pc.qualitative.Safe # couleur cohérente par cluster

In [164]:
from csv import list_dialects
# mapbox avec 4 cartes sous logiques
# liste deroulante heure = for i in range(0, 23)
# list_deroulante = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig = make_subplots(
    rows=2,
    cols=4,
    subplot_titles=[
        "Thu 22h", "Fri 22h", "Sat 22h", "Sun 20h",
        "Thu 22h", "Fri 22h", "Sat 22h", "Sun 20h",
    ],
    specs=[
        [{"type": "mapbox"}]*4,
        [{"type": "xy"}]*4
    ],
)



In [165]:
result=[]
mapbox_ids = ["mapbox", "mapbox2", "mapbox3", "mapbox4"]

for i, res in enumerate(results):
    fig.add_trace(
        go.Scattermapbox(
            lat=res["df"]["Lat"],
            lon=res["df"]["Lon"],
            mode="markers",
            marker=dict(
                size=5,
                color=res["df"]["Cluster_KMeans"],
                colorscale="Viridis",
                opacity=0.6,
            ),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )

    fig.add_trace(
        go.Scattermapbox(
            lat=res["centroids"]["Lat"],
            lon=res["centroids"]["Lon"],
            mode="markers",
            marker=dict(size=14, color="black"),
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )


In [166]:
for i, res in enumerate(results):
    fig.add_trace(
        go.Scatter(
            x=res["df"]["Lon"],
            y=res["df"]["Lat"],

            mode="markers",
            marker=dict(
                size=4,
                color=res["df"]["Cluster_KMeans"],
                colorscale="Viridis",
                opacity=0.6,
            ),
            showlegend=False,
        ),
        row=2,
        col=i + 1,
    )


In [167]:
fig.update_layout(
    height=900,
    title="Uber Hot-Zones Comparison – Multiple Time Slots",

    mapbox=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.75, lon=-73.98),
        domain=dict(x=[0.00, 0.25], y=[0.55, 1.00]),
    ),

    mapbox2=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.75, lon=-73.98),
        domain=dict(x=[0.25, 0.50], y=[0.55, 1.00]),
    ),

    mapbox3=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.75, lon=-73.98),
        domain=dict(x=[0.50, 0.75], y=[0.55, 1.00]),
    ),

    mapbox4=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.75, lon=-73.98),
        domain=dict(x=[0.75, 1.00], y=[0.55, 1.00]),
    ),
)


# Pipeline multi-mois

In [168]:

files = [
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-apr14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-may14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-jun14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-jul14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-aug14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-sep14.csv",
    "/content/drive/MyDrive/JedhaBootcamp/uber-raw-data-janjune-15.csv",
]

dfs = [pd.read_csv(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) # ignore_index=True est important pour éviter des index dupliqués

df_all.shape


(18804806, 8)

In [169]:
df_all.info()
df_all.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18804806 entries, 0 to 18804805
Data columns (total 8 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Date/Time             object 
 1   Lat                   float64
 2   Lon                   float64
 3   Base                  object 
 4   Dispatching_base_num  object 
 5   Pickup_date           object 
 6   Affiliated_base_num   object 
 7   locationID            float64
dtypes: float64(3), object(5)
memory usage: 1.1+ GB


,Date/Time,Lat,Lon,Base,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512,NaN,NaN,NaN,NaN
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512,NaN,NaN,NaN,NaN
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512,NaN,NaN,NaN,NaN
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512,NaN,NaN,NaN,NaN
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512,NaN,NaN,NaN,NaN


In [170]:
df_all.tail()



,Date/Time,Lat,Lon,Base,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
18804801,NaN,NaN,NaN,NaN,B02765,2015-05-08 15:43:00,B02765,186.0
18804802,NaN,NaN,NaN,NaN,B02765,2015-05-08 15:43:00,B02765,263.0
18804803,NaN,NaN,NaN,NaN,B02765,2015-05-08 15:43:00,B02765,90.0
18804804,NaN,NaN,NaN,NaN,B02765,2015-05-08 15:44:00,B01899,45.0
18804805,NaN,NaN,NaN,NaN,B02765,2015-05-08 15:44:00,B02682,144.0


In [171]:
df

,Date/Time,Lat,Lon,Base,weekday,hour
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,Tuesday,0
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,Tuesday,0
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,Tuesday,0
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,Tuesday,0
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,Tuesday,0
...,...,...,...,...,...,...
564511,2014-04-30 23:22:00,40.7640,-73.9744,B02764,Wednesday,23
564512,2014-04-30 23:26:00,40.7629,-73.9672,B02764,Wednesday,23
564513,2014-04-30 23:31:00,40.7443,-73.9889,B02764,Wednesday,23
564514,2014-04-30 23:32:00,40.6756,-73.9405,B02764,Wednesday,23


In [172]:
df_all.describe(include="all")

,Date/Time,Lat,Lon,Base,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
count,4534327,4.534327e+06,4.534327e+06,4534327,14270479,14270479,14108284,1.427048e+07
unique,260093,NaN,NaN,5,8,2744783,284,NaN
top,4/7/2014 20:21:00,NaN,NaN,B02617,B02764,2015-06-27 20:52:00,B02764,NaN
freq,97,NaN,NaN,1458853,5753653,213,4352321,NaN
mean,NaN,4.073926e+01,-7.397302e+01,NaN,NaN,NaN,NaN,1.520574e+02
std,NaN,3.994991e-02,5.726670e-02,NaN,NaN,NaN,NaN,7.159620e+01
min,NaN,3.965690e+01,-7.492900e+01,NaN,NaN,NaN,NaN,1.000000e+00
25%,NaN,4.072110e+01,-7.399650e+01,NaN,NaN,NaN,NaN,9.200000e+01
50%,NaN,4.074220e+01,-7.398340e+01,NaN,NaN,NaN,NaN,1.570000e+02
75%,NaN,4.076100e+01,-7.396530e+01,NaN,NaN,NaN,NaN,2.300000e+02


In [173]:
df_all.isna().sum().sum()

np.int64(75381419)

# réduction de la taille du dataset et réduction aux strictes colonnes nécessaires après suppression des valeurs manquantes

In [174]:
cols_needed = ["Date/Time", "Lat", "Lon"]

df_all_clean = df_all[cols_needed].copy()


In [175]:
df_all_clean = df_all_clean.dropna(subset=["Lat", "Lon"])


In [176]:
df_all_clean["Date/Time"] = pd.to_datetime(df_all_clean["Date/Time"])
df_all_clean["weekday"] = df_all_clean["Date/Time"].dt.day_name()
df_all_clean["hour"] = df_all_clean["Date/Time"].dt.hour


In [177]:
df_all_clean.info()
df_all_clean.isna().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 4534327 entries, 0 to 4534326
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   Date/Time  datetime64[ns]
 1   Lat        float64       
 2   Lon        float64       
 3   weekday    object        
 4   hour       int32         
dtypes: datetime64[ns](1), float64(2), int32(1), object(1)
memory usage: 190.3+ MB


,0
Date/Time,0
Lat,0
Lon,0
weekday,0
hour,0


In [178]:
#h= 0
# df
time_slots = [
    {"label": "Mon_20h", "weekday": "Monday", "hour": 20}, # là il manque le mois
    {"label": "Tue_20h", "weekday": "Tuesday", "hour": 20},
    {"label": "Wed_20h", "weekday": "Wednesday", "hour": 20},
    {"label": "Thu_22h", "weekday": "Thursday", "hour": 22},
    {"label": "Fri_22h", "weekday": "Friday",   "hour": 22}, # [hour = h for h in range (0, 23)]
    {"label": "Sat_22h", "weekday": "Saturday", "hour": 22},
    {"label": "Sun_20h", "weekday": "Sunday",   "hour": 20},
]


In [179]:
features = ["Lat", "Lon"]

def build_clusters(df, weekday, hour, k=7):
    df_focus = df[(df["weekday"] == weekday) & (df["hour"] == hour)].copy()

    kmeans = KMeans(n_clusters=k, random_state=42)
    df_focus["Cluster_KMeans"] = kmeans.fit_predict(df_focus[features]) + 1

    centroids = (
        df_focus
        .groupby("Cluster_KMeans")[features]
        .mean()
        .reset_index()
    )

    return df_focus, centroids


In [180]:
# results = []

for slot in time_slots:
    output = process_time_slot(
        df=df,
        weekday=slot["weekday"],
        hour=slot["hour"],
        label=slot["label"],
        k=7
    )
    if output:
        results.append(output)


In [181]:
results_all = {}

for slot in time_slots:
    df_slot, centroids = build_clusters(
        df_all_clean,
        weekday=slot["weekday"],
        hour=slot["hour"],
        k=7
    )

    results_all[slot["label"]] = {
        "df": df_slot,
        "centroids": centroids
    }

    print(f"{slot['label']} → {df_slot.shape[0]} pickups")


Mon_20h → 32849 pickups
Tue_20h → 44661 pickups
Wed_20h → 47772 pickups
Thu_22h → 44194 pickups
Fri_22h → 49409 pickups
Sat_22h → 47951 pickups
Sun_20h → 25076 pickups


In [182]:
# mapbox avec 7 cartes sous logiques
fig = make_subplots(
    rows=4,
    cols=4,
    subplot_titles=[
        "Mon 20h", "Tue 20h","Wed 20h", "Thu 22h", "Fri 22h", "Sat 22h", "Sun 20h",
        "Mon 20h", "Tue 20h","Wed 20h", "Thu 22h", "Fri 22h", "Sat 22h", "Sun 20h",
    ],
    specs=[ # attend une matrice 4*4
        [{"type": "mapbox"}]*4,# 4 pour 4 colonnes
        [{"type": "mapbox"}]*4,
        [{"type": "xy"}]*4,
        [{"type": "xy"}]*4
    ],
)

In [183]:

#  mapbox_ids = ["mapbox", "mapbox2", "mapbox3", "mapbox4", "mapbox5", "mapbox6", "mapbox7"]

# for i, (label, res) in enumerate(results_all.items()):
#     if i >= len(mapbox_ids):
#         break  # on remplit uniquement la 1ʳᵉ ligne (4 colonnes)

#     fig.add_trace(
#         go.Scattermapbox(
#             lat=res["df"]["Lat"],
#             lon=res["df"]["Lon"],
#             mode="markers",
#             marker=dict(
#                 size=4,
#                 color=res["df"]["Cluster_KMeans"],
#                 colorscale="Viridis",
#                 opacity=0.6,
#             ),
#             showlegend=False,
#             subplot=mapbox_ids[i],  #  LA LIGNE CLÉ!!!!!
#         ),
#         row=1 if i < 4 else 2,
#         col=i + 1 if i < 4 else i - 3,
#     )

#     fig.add_trace(
#         go.Scattermapbox(
#             lat=res["centroids"]["Lat"],
#             lon=res["centroids"]["Lon"],
#             mode="markers",
#             marker=dict(size=14, color="black"),
#             showlegend=False,
#             subplot=mapbox_ids[i],  #  LA LIGNE CLÉ!!!!!
#         ),
#         row=1 if i < 4 else 2,
#         col=i + 1 if i < 4 else i - 3,
#     )

for i, (label, res) in enumerate(results_all.items()):
    if i >= 7:
        break

    row = 1 if i < 4 else 2
    col = i + 1 if i < 4 else i - 3

    fig.add_trace(
        go.Scattermapbox(
            lat=res["df"]["Lat"],
            lon=res["df"]["Lon"],
            mode="markers",
            marker=dict(
                size=4,
                color=res["df"]["Cluster_KMeans"],
                colorscale="Viridis",
                opacity=0.6,
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )


In [184]:
# for i, (label, res) in enumerate(results_all.items()):
#     row = 3 if i < 4 else 4
#     col = i + 1 if i < 4 else i - 3

#     fig.add_trace(
#         go.Scatter(
#             x=res["df"]["Lon"],
#             y=res["df"]["Lat"],
#             mode="markers",
#             marker=dict(
#                 size=4,
#                 color=res["df"]["Cluster_KMeans"],
#                 colorscale="Viridis",
#                 opacity=0.6,
#             ),
#             showlegend=False,
#         ),
#         row=row,
#         col=col,
#     )
for i, (label, res) in enumerate(results_all.items()):
    if i >= 7:
        break

    row = 3 if i < 4 else 4
    col = i + 1 if i < 4 else i - 3

    fig.add_trace(
        go.Scatter(
            x=res["df"]["Lon"],
            y=res["df"]["Lat"],
            mode="markers",
            marker=dict(
                size=4,
                color=res["df"]["Cluster_KMeans"],
                colorscale="Viridis",
                opacity=0.6,
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )



In [185]:
print (len(results_all.items()))

7


In [186]:
fig.update_layout(
    height=900,
    title="Uber Hot-Zones Comparison – Multiple Time Slots",
    mapbox_style="carto-positron",
    mapbox_center=dict(lat=40.75, lon=-73.98),
    mapbox_zoom=10,
)


Output hidden; open in https://colab.research.google.com to view.




    # mapbox=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.00, 0.25], y=[0.55, 1.00]),
    # ),

    # mapbox2=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.25, 0.50], y=[0.55, 1.00]),
    # ),

    # mapbox3=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.50, 0.75], y=[0.55, 1.00]),
    # ),

    # mapbox4=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.75, 1.00], y=[0.55, 1.00]),
    # ),
    #  mapbox5=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.75, 1.00], y=[0.55, 1.00]),
    # ),
    #  mapbox6=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.75, 1.00], y=[0.55, 1.00]),
    # ),
    #  mapbox7=dict(
    #     style="carto-positron",
    #     zoom=10,
    #     center=dict(lat=40.75, lon=-73.98),
    #     domain=dict(x=[0.75, 1.00], y=[0.55, 1.00]),
    # ),

